# 01 — Обучение CNN14 (PANNs) на GTZAN

**Что делает ноутбук:**
- Запускает 5-fold × 3 seeds = **15 runs** обучения CNN14 с PANNs-предобучением.
- Two-stage progressive unfreezing: stage 1 (head only) + stage 2 (conv_block6 + fc1 + head).
- AdamW + дифференцированный lr (backbone 1e-4, head 1e-3), Cosine + warmup, AMP fp16.
- SpecAugment + Mixup α=0.2.
- Сохраняет best checkpoint по val macro-F1 для каждого run.

**Входы:**
- `/kaggle/input/gtzan-cnn14-resnet18-src/src/` — наш код.
- `/kaggle/input/gtzan-preproc/gtzan_logmel.h5` + `folds.json` — результат Notebook 00.
- `/kaggle/input/cnn14-panns/Cnn14_mAP=0.431.pth` — PANNs checkpoint (загрузить как Kaggle Dataset либо wget внутри).

**Выходы (`/kaggle/working/outputs/`):**
- `best_cnn14_fold{F}_seed{S}.pth` × 15.
- `cnn14_metrics.csv` — все метрики 15 runs.
- `cnn14_history.csv` — per-epoch history.
- `runs/` — TensorBoard logs.

**Время на T4:** ~6-8 часов на 15 runs (зависит от early stopping).

In [ ]:
!pip install -q librosa==0.10.1 h5py soxr fvcore grad-cam umap-learn 2>&1 | tail -3

In [ ]:
import sys
from pathlib import Path

for p in ['/kaggle/input/gtzan-cnn14-resnet18-src', '/kaggle/input/cnn14-resnet18-src',
          '/kaggle/working', str(Path.cwd().parent)]:
    if Path(p, 'src', '__init__.py').exists():
        sys.path.insert(0, p)
        print(f'src/ at: {p}')
        break

import torch
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name() if torch.cuda.is_available() else "cpu"}')

In [ ]:
# Поиск артефактов
import os

def find_file(*candidates):
    for c in candidates:
        if Path(c).exists():
            return Path(c)
    return None

H5_PATH = find_file(
    '/kaggle/input/gtzan-preproc/gtzan_logmel.h5',
    '/kaggle/working/gtzan_logmel.h5',
    Path.cwd().parent / 'outputs' / 'gtzan_logmel.h5',
)
FOLDS_PATH = find_file(
    '/kaggle/input/gtzan-preproc/folds.json',
    '/kaggle/working/folds.json',
    Path.cwd().parent / 'outputs' / 'folds.json',
)
CNN14_CKPT = find_file(
    '/kaggle/input/cnn14-panns/Cnn14_mAP=0.431.pth',
    '/kaggle/working/Cnn14_mAP=0.431.pth',
    Path.cwd().parent / 'models' / 'Cnn14_mAP=0.431.pth',
)

# Если PANNs checkpoint не найден — скачиваем (требует internet в Kaggle settings)
if CNN14_CKPT is None:
    out = Path('/kaggle/working/Cnn14_mAP=0.431.pth')
    print(f'Скачиваем PANNs Cnn14 (~340 MB)...')
    os.system(f'wget -q -O {out} "https://zenodo.org/record/3987831/files/Cnn14_mAP%3D0.431.pth"')
    CNN14_CKPT = out

assert H5_PATH and H5_PATH.exists(), 'gtzan_logmel.h5 не найден. Запустите 00_data_prep.ipynb.'
assert FOLDS_PATH and FOLDS_PATH.exists(), 'folds.json не найден.'
assert CNN14_CKPT and CNN14_CKPT.exists(), 'Cnn14_mAP=0.431.pth не найден.'
print(f'H5:    {H5_PATH}')
print(f'Folds: {FOLDS_PATH}')
print(f'CNN14: {CNN14_CKPT} ({CNN14_CKPT.stat().st_size / 1e6:.1f} MB)')

OUT_DIR = Path('/kaggle/working/outputs')
OUT_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
import json
import time
import pandas as pd

from src.configs import TrainConfig, AugConfig, DEFAULT_SEEDS, N_FOLDS
from src.dataset import load_folds
from src.models import build_model
from src.train import fit_two_stage, build_loaders

folds = load_folds(FOLDS_PATH)

# Базовый TrainConfig для CNN14
def make_cfg(fold: int, seed: int) -> TrainConfig:
    return TrainConfig(
        model_name='cnn14',
        fold=fold,
        seed=seed,
        run_name=f'cnn14_f{fold}_s{seed}',
        batch_size=32,
        stage1_epochs=10,
        stage2_epochs=30,
        early_stop_patience=7,
        lr_backbone=1e-4,
        lr_head=1e-3,
        wd_backbone=1e-4,
        wd_head=1e-3,
        warmup_epochs=2,
        label_smoothing=0.1,
        grad_clip_norm=1.0,
        use_amp=True,
        aug=AugConfig(use_specaugment=True, use_mixup=True, mixup_alpha=0.2),
        h5_path=str(H5_PATH),
        folds_json_path=str(FOLDS_PATH),
        cnn14_ckpt_path=str(CNN14_CKPT),
        output_dir=str(OUT_DIR),
        num_workers=2,
    )

print(f'Запланировано runs: {N_FOLDS} folds × {len(DEFAULT_SEEDS)} seeds = {N_FOLDS * len(DEFAULT_SEEDS)}')

In [ ]:
# Главный цикл: 5 folds × 3 seeds. Можно прервать и продолжить — best_*.pth кешируются.
all_results = []
all_histories = []
global_start = time.time()

for fold in range(N_FOLDS):
    fold_data = folds[str(fold)]
    train_ids, val_ids = fold_data['train'], fold_data['val']
    print(f'\n========== FOLD {fold} (train={len(train_ids)} val={len(val_ids)}) ==========')
    for seed in DEFAULT_SEEDS:
        cfg = make_cfg(fold, seed)
        run_dir = OUT_DIR / 'runs' / cfg.run_name
        # Пропуск, если best уже есть и завершён
        best_path = OUT_DIR / f'best_cnn14_fold{fold}_seed{seed}.pth'
        if best_path.exists():
            ckpt = torch.load(best_path, map_location='cpu')
            if ckpt.get('stage', 0) == 2:
                print(f'[skip] {cfg.run_name} уже обучен (best val_f1={ckpt["val_f1"]:.3f})')
                all_results.append({'fold': fold, 'seed': seed, 'best_val_f1': ckpt['val_f1']})
                continue

        print(f'\n>>> {cfg.run_name}')
        model = build_model('cnn14', pann_ckpt_path=str(CNN14_CKPT))
        train_loader, val_loader = build_loaders(str(H5_PATH), train_ids, val_ids, cfg)

        result = fit_two_stage(model, train_loader, val_loader, cfg, out_dir=OUT_DIR, tb_dir=run_dir)
        for row in result['history']:
            all_histories.append({'model': 'cnn14', 'fold': fold, 'seed': seed, **row})
        all_results.append({
            'fold': fold, 'seed': seed,
            'best_val_f1': result['best_val_f1'],
            'best_epoch': result['best_epoch'],
            'best_path': result['best_path'],
        })

        # Промежуточное сохранение метрик
        pd.DataFrame(all_histories).to_csv(OUT_DIR / 'cnn14_history.csv', index=False)
        pd.DataFrame(all_results).to_csv(OUT_DIR / 'cnn14_metrics.csv', index=False)
        # Освобождаем GPU
        del model, train_loader, val_loader
        torch.cuda.empty_cache()

        elapsed_h = (time.time() - global_start) / 3600
        print(f'[total elapsed] {elapsed_h:.2f}h')

In [ ]:
# Итоговая сводка
df = pd.DataFrame(all_results)
print(df.to_string(index=False))
if not df.empty:
    print(f'\nCNN14 best val_f1: mean={df["best_val_f1"].mean():.3f} ± {df["best_val_f1"].std():.3f}')

In [ ]:
# Освобождаем место в Kaggle output: удаляем 'last' чекпойнты, оставляем только best
for p in OUT_DIR.glob('*.pth'):
    if 'best_' not in p.name:
        p.unlink()
        print(f'rm {p.name}')

# Уменьшаем размер best: убираем optimizer state (если был)
for p in OUT_DIR.glob('best_*.pth'):
    ckpt = torch.load(p, map_location='cpu')
    if 'optimizer' in ckpt:
        del ckpt['optimizer']
        torch.save(ckpt, p)
print('Cleanup done')